# Final Mini-Project Solution: A Toy Resonance Search

This notebook implements a simplified resonance-search workflow using:

- NumPy for toy-data generation
- Matplotlib and mplhep for visualization
- iminuit for fitting
- pyhf for statistical inference

The goal is to search for a narrow Gaussian signal peak on top of a smooth falling background.

## Learning goals

By the end of this notebook, you will be able to:

1. Generate a toy signal-plus-background mass spectrum
2. Fill and visualize an invariant-mass histogram
3. Fit a signal-plus-background model with `iminuit`
4. Define a signal region and estimate expected yields
5. Build a one-bin `pyhf` model
6. Fit the signal-strength parameter
7. Test a signal hypothesis

## Setup

Install the required packages if needed:

```bash
pip install numpy matplotlib mplhep iminuit pyhf
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from iminuit import Minuit
from iminuit.cost import LeastSquares

import pyhf

try:
    import mplhep as hep
except ImportError:
    hep = None

# 1. Generate a toy mass spectrum

We generate:

- a smooth exponential background
- a Gaussian signal centered at 125 GeV

The signal is intentionally exaggerated so that it is visible in a short tutorial.

In [ ]:
rng = np.random.default_rng(42)

mass_min = 80.0
mass_max = 170.0

n_background = 20_000
n_signal = 1_000

background = rng.exponential(scale=35.0, size=n_background) + 50.0
background = background[
    (background >= mass_min) & (background <= mass_max)
]

signal = rng.normal(
    loc=125.0,
    scale=2.0,
    size=n_signal,
)
signal = signal[
    (signal >= mass_min) & (signal <= mass_max)
]

mass = np.concatenate([background, signal])

print(f"Background events: {len(background):,}")
print(f"Signal events:     {len(signal):,}")
print(f"Total events:      {len(mass):,}")

# 2. Fill and plot the invariant-mass histogram

In [ ]:
n_bins = 45

counts, edges = np.histogram(
    mass,
    bins=n_bins,
    range=(mass_min, mass_max),
)

centers = 0.5 * (edges[:-1] + edges[1:])
bin_width = edges[1] - edges[0]

if hep is not None:
    plt.style.use(hep.style.CMS)

plt.step(
    edges[:-1],
    counts,
    where="post",
    label="Toy data",
)

plt.xlabel("Invariant mass [GeV]")
plt.ylabel(f"Events / {bin_width:.1f} GeV")
plt.title("Toy resonance search")
plt.legend()
plt.show()

# 3. Define the fit model

We model the spectrum as

- an exponential background
- a Gaussian signal

The Gaussian amplitude represents the peak height, not the total number of signal events.

In [ ]:
def signal_plus_background(
    x,
    background_norm,
    background_slope,
    signal_amplitude,
    signal_mean,
    signal_sigma,
):
    background_component = background_norm * np.exp(
        -background_slope * (x - mass_min)
    )

    signal_component = signal_amplitude * np.exp(
        -0.5 * ((x - signal_mean) / signal_sigma) ** 2
    )

    return background_component + signal_component

For this tutorial, we use a weighted least-squares fit with Poisson-like uncertainties:

```python
errors = sqrt(max(counts, 1))
```

A full analysis would usually use a binned Poisson likelihood instead.

In [ ]:
errors = np.sqrt(np.maximum(counts, 1.0))

least_squares = LeastSquares(
    centers,
    counts,
    errors,
    signal_plus_background,
)

fit = Minuit(
    least_squares,
    background_norm=float(counts.max()),
    background_slope=0.025,
    signal_amplitude=300.0,
    signal_mean=125.0,
    signal_sigma=2.0,
)

fit.limits["background_norm"] = (0, None)
fit.limits["background_slope"] = (0, None)
fit.limits["signal_amplitude"] = (0, None)
fit.limits["signal_mean"] = (120.0, 130.0)
fit.limits["signal_sigma"] = (0.2, 10.0)

fit.migrad()
fit.hesse()

fit

In [ ]:
for name in fit.parameters:
    print(
        f"{name:20s} = "
        f"{fit.values[name]:10.4f} ± {fit.errors[name]:.4f}"
    )

# 4. Plot the fitted signal and background components

In [ ]:
def background_model(x, norm, slope):
    return norm * np.exp(-slope * (x - mass_min))


def gaussian_model(x, amplitude, mean, sigma):
    return amplitude * np.exp(
        -0.5 * ((x - mean) / sigma) ** 2
    )


x_plot = np.linspace(mass_min, mass_max, 1_000)

background_fit = background_model(
    x_plot,
    fit.values["background_norm"],
    fit.values["background_slope"],
)

signal_fit = gaussian_model(
    x_plot,
    fit.values["signal_amplitude"],
    fit.values["signal_mean"],
    fit.values["signal_sigma"],
)

total_fit = background_fit + signal_fit

plt.errorbar(
    centers,
    counts,
    yerr=errors,
    fmt="o",
    label="Toy data",
)

plt.plot(
    x_plot,
    total_fit,
    label="Signal + background fit",
)

plt.plot(
    x_plot,
    background_fit,
    linestyle="--",
    label="Background component",
)

plt.plot(
    x_plot,
    signal_fit,
    linestyle=":",
    label="Signal component",
)

plt.xlabel("Invariant mass [GeV]")
plt.ylabel(f"Events / {bin_width:.1f} GeV")
plt.legend()
plt.show()

# 5. Define a signal region

We define a signal region centered on the fitted peak:

\[
m_\mathrm{fit} - 2\sigma_\mathrm{fit}
<
m
<
m_\mathrm{fit} + 2\sigma_\mathrm{fit}.
\]

We then estimate:

- observed events in the region
- expected background yield
- expected signal yield

In [ ]:
mean_fit = fit.values["signal_mean"]
sigma_fit = fit.values["signal_sigma"]

signal_region_low = mean_fit - 2.0 * sigma_fit
signal_region_high = mean_fit + 2.0 * sigma_fit

print(
    f"Signal region: "
    f"{signal_region_low:.2f} < m < {signal_region_high:.2f} GeV"
)

In [ ]:
observed_signal_region = np.count_nonzero(
    (mass >= signal_region_low)
    & (mass < signal_region_high)
)

print("Observed events in signal region:", observed_signal_region)

To estimate the fitted signal and background yields, integrate the model components over the signal region and divide by the histogram bin width.

In [ ]:
integration_grid = np.linspace(
    signal_region_low,
    signal_region_high,
    10_000,
)

fitted_background_density = background_model(
    integration_grid,
    fit.values["background_norm"],
    fit.values["background_slope"],
)

fitted_signal_density = gaussian_model(
    integration_grid,
    fit.values["signal_amplitude"],
    fit.values["signal_mean"],
    fit.values["signal_sigma"],
)

expected_background_sr = (
    np.trapz(fitted_background_density, integration_grid)
    / bin_width
)

expected_signal_sr = (
    np.trapz(fitted_signal_density, integration_grid)
    / bin_width
)

print(f"Expected background yield: {expected_background_sr:.2f}")
print(f"Expected signal yield:     {expected_signal_sr:.2f}")
print(f"Observed yield:            {observed_signal_region}")

# 6. Build a one-bin `pyhf` model

We use the signal-region yields to construct a one-bin counting model.

For illustration, assume a 10% uncertainty on the fitted background yield.

In [ ]:
background_uncertainty = 0.10 * expected_background_sr

model = pyhf.simplemodels.uncorrelated_background(
    signal=[expected_signal_sr],
    bkg=[expected_background_sr],
    bkg_uncertainty=[background_uncertainty],
)

data = [float(observed_signal_region)] + model.config.auxdata

print("Model parameters:", model.config.par_names)
print("Observed data:", data)

# 7. Fit the signal strength

The parameter of interest is the signal strength, \(\mu\):

- \(\mu = 0\): background-only
- \(\mu = 1\): nominal fitted signal

In [ ]:
best_fit = pyhf.infer.mle.fit(data, model)

mu_hat = float(best_fit[model.config.poi_index])

print("Best-fit parameters:", best_fit)
print(f"Best-fit signal strength: mu_hat = {mu_hat:.3f}")

# 8. Test a signal hypothesis

We test the nominal signal hypothesis, \(\mu = 1\), using the modified frequentist CLs construction.

In [ ]:
test_mu = 1.0

cls_obs = pyhf.infer.hypotest(
    test_mu,
    data,
    model,
    test_stat="qtilde",
)

print(f"Observed CLs for mu={test_mu}: {float(cls_obs):.6f}")

A smaller CLs value means that the tested signal hypothesis is less compatible with the observed data.

This is not the same as a discovery significance. A discovery test typically evaluates the background-only hypothesis with a dedicated test statistic.

# 9. Compare several signal hypotheses

In [ ]:
test_values = [0.0, 0.5, 1.0, 1.5, 2.0]

for mu in test_values:
    try:
        result = pyhf.infer.hypotest(
            mu,
            data,
            model,
            test_stat="qtilde",
        )
        print(f"mu={mu:3.1f}  CLs={float(result):.6f}")
    except Exception as err:
        print(f"mu={mu:3.1f}  unavailable: {err}")

# 10. Summary of the result

In [ ]:
print("Toy resonance-search summary")
print("----------------------------")
print(f"Fitted peak position:       {mean_fit:.3f} GeV")
print(f"Fitted peak width:          {sigma_fit:.3f} GeV")
print(f"Signal-region observation:  {observed_signal_region}")
print(f"Expected background:        {expected_background_sr:.2f}")
print(f"Expected signal:            {expected_signal_sr:.2f}")
print(f"Best-fit signal strength:   {mu_hat:.3f}")
print(f"CLs for mu=1:               {float(cls_obs):.6f}")

# Optional extension 1: Add a background-shape uncertainty

A one-bin counting model cannot represent shape information directly.

To include a background-shape uncertainty:

1. Divide the mass spectrum into several bins
2. Define nominal signal and background yields in each bin
3. Add a `histosys` or `shapesys` modifier to the background sample
4. Build the full HistFactory-style workspace specification

# Optional extension 2: Scan over resonance masses

Repeat the fit for a grid of signal-mass hypotheses:

```python
mass_hypotheses = np.arange(110.0, 141.0, 1.0)
```

At each mass:

1. Fix or initialize the Gaussian mean
2. Refit the remaining parameters
3. Store the fitted signal amplitude or test statistic
4. Plot the scan as a function of the mass hypothesis

# Optional extension 3: Use Awkward Array

For event-level data with variable-length particle collections, replace flat NumPy arrays with Awkward Arrays.

A typical pipeline is:

```text
Uproot
  → Awkward Array
  → object selection
  → invariant-mass calculation
  → histogram
  → fit
  → statistical inference
```

# Optional extension 4: Read a ROOT file with Uproot

Replace the toy generator with a ROOT input:

```python
import uproot

with uproot.open("events.root:Events") as tree:
    arrays = tree.arrays(
        ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass"]
    )
```

You can then construct four-vectors, select candidate events, and fill the invariant-mass histogram from real or simulated event data.

# Final remarks

This notebook demonstrates the main conceptual stages of a resonance search:

- build an observable
- model signal and background
- estimate parameters
- define a signal-sensitive region
- construct a likelihood
- fit the signal strength
- test a signal hypothesis

The statistical model here is deliberately simplified. Real HEP analyses use multi-bin likelihoods, correlated systematic uncertainties, validation regions, and more sophisticated signal and background models.